## Importer guide and examples

The importer turns flat rows (`ImportRow`) into a consistent set of ORM entities:

`Project → Patient → Study → Series → ImageInstance → ImageStorage` (+ device + storage backend + scan + modality)

A typical workflow is:

- Create a list of `ImportRow` objects (or load them from a CSV with `read_import_rows_csv`)
- Call `plan_image_import(session, rows, ...)` to produce an `ImportRun`
- Inspect the planned changes (`run.display_summary()` / `run.full_summary()`)
- Apply them (`run.apply()` + `session.commit()`)
- Optionally write the run to JSON (`run.write_json(...)`) and later load it (`ImportRun.read_json(...)`)
- Optionally undo a completed run (`run.undo()` + commit)

Notes:

- `plan_image_import` / `plan_import` may mutate ORM objects in-memory during planning; nothing is persisted until `run.apply()` and `commit()`.
- `run.apply()` writes to the database but does not commit.

In [ ]:
from datetime import datetime
from pathlib import Path

from sqlalchemy import select, func
from sqlalchemy.orm import Session

from eyened_orm.importer import plan_image_import, ImportRow, ImportRun
from eyened_orm.importer.import_csv import read_import_rows_csv

from eyened_orm import (
    Project,
    Patient,
    Study,
    Series,
    ImageInstance,
    ImageStorage,
    ModalityTable,
    Scan,
)


# Helper: creates an in-memory SQLite DB compatible with the EyeNED ORM schema.
from eyened_orm.utils.sqlite_testdb import create_sqlite_memory_sessionmaker

SessionLocal = create_sqlite_memory_sessionmaker(expire_on_commit=False)
print("Ready: in-memory database created")


def count(session: Session, model) -> int:
    return session.scalar(select(func.count()).select_from(model))


def print_counts(session: Session):
    for model in (
        Project,
        Patient,
        Study,
        Series,
        ImageInstance,
        ImageStorage,
        ModalityTable,
        Scan,
    ):
        print(f"  {model.__name__:<15} : {count(session, model)}")


# Create a fresh session for the examples below.
session = SessionLocal()
print("Session ready")

Ready: in-memory database created
Session ready


## Load `ImportRow` records from a CSV

If you already have your metadata in a CSV, you can load it directly into `ImportRow` objects and feed them into `plan_image_import`.

The CSV header should use `ImportRow` field names (snake_case), for example: `project_name`, `patient_identifier`, `study_date`, `public_id`, `object_key`, `modality_tag`, `scan_mode`, ...

In [2]:
csv_defaults = {
    "project_external": "Y",
    "manufacturer": "UNKOWN",
    "manufacturer_model_name": "UNKOWN",
    "device_description": "UNKOWN",
    "dataset_identifier": "",  # deprecated, but required by schema
    "storage_backend_kind": "local",
}

csv_path = Path("./_importer_output/example_rows.csv")
rows_csv = read_import_rows_csv(csv_path)

print(f"Loaded {len(rows_csv)} rows from {csv_path}")

session_csv = SessionLocal()
run_csv = plan_image_import(session_csv, rows_csv, defaults=csv_defaults)
run_csv.display_summary()
run_csv.apply()
session_csv.commit()

print("Counts after CSV import:")
print_counts(session_csv)

Loaded 2 rows from _importer_output/example_rows.csv


Entity,Update,Create,Total
Series,,2,2
ImageInstance,,2,2
ImageStorage,,2,2
StorageBackend,,1,1
Scan,,1,1
ModalityTable,,1,1
DeviceModel,,1,1
Project,,1,1
DeviceInstance,,1,1
Patient,,1,1


Counts after CSV import:
  Project         : 1
  Patient         : 1
  Study           : 1
  Series          : 2
  ImageInstance   : 2
  ImageStorage    : 2
  ModalityTable   : 1
  Scan            : 1


## Import new images (+ related entities)

Typical case: you have new image files and want the importer to create (or reuse) the required hierarchy.

We’ll build a few `ImportRow`, plan + review, then apply + commit.


In [3]:
defaults = {
    # Commonly required / convenient defaults
    "project_external": "Y",
    "manufacturer": "UNKOWN",
    "manufacturer_model_name": "UNKOWN",
    "device_description": "UNKOWN",
    "dataset_identifier": "",  # deprecated, but required by schema
    "storage_backend_kind": "local",
}

rows = [
    ImportRow(
        project_name="demo-project",
        storage_backend_key="demo-backend",
        object_key=f"demo-{i}.png",
        patient_identifier="patient-1",
        study_date=datetime.now().date(),
        modality="ColorFundus",
        laterality="L",
        modality_tag="notebook_vendor_modality",
        scan_mode="notebook_demo_scan",
    )
    for i in range(2)
]

run = plan_image_import(session, rows, defaults=defaults)
run.display_summary()

run.apply()
session.commit()

print("Counts after import:")
print_counts(session)

Entity,Update,Create,Total
Series,,2,2
ImageInstance,,2,2
ImageStorage,,2,2
StorageBackend,,1,1
Scan,,1,1
ModalityTable,,1,1
Project,,1,1
Patient,,1,1
Study,,1,1


Counts after import:
  Project         : 2
  Patient         : 2
  Study           : 2
  Series          : 4
  ImageInstance   : 4
  ImageStorage    : 4
  ModalityTable   : 2
  Scan            : 2


### Example: attach to existing objects

When part of the hierarchy already exists, you can attach to it by using either primary keys (`*_id`) or the usual lookup fields (e.g. `project_name` + `patient_identifier` + `study_date`).

In [4]:
rows = [
    ImportRow(
        project_name="demo-project",  # lookup existing project by name
        patient_identifier="patient-1",  # lookup existing patient by project_name + patient_identifier
        study_date=datetime.now().date(),  # lookup existing study by project_name + patient_identifier + study_date
        modality="ColorFundus",  # modality of new anonymous image
        laterality="L",  # laterality of new anonymous image
        modality_tag="notebook_vendor_modality",
        scan_mode="notebook_demo_scan",
        storage_backend_key="demo-backend",  # lookup existing storage backend by key
        object_key=f"new_image.png",  # new image storage
    )
    for i in range(2)
]

run = plan_image_import(session, rows, defaults=defaults)
run.display_summary()

run.apply()
session.commit()

print("Counts after import:")
print_counts(session)

Entity,Update,Create,Total
Series,,1,1
ImageInstance,,1,1
ImageStorage,,1,1


Counts after import:
  Project         : 2
  Patient         : 2
  Study           : 2
  Series          : 5
  ImageInstance   : 5
  ImageStorage    : 5
  ModalityTable   : 2
  Scan            : 2


### Example: group images into the same Series with `series_anonymous_identity`

Sometimes you don’t know the real lookup key for an entity (e.g. no `series_instance_uid`). The importer supports *batch-local grouping* via `*_anonymous_identity` fields. 
For example, rows with the same `series_anonymous_identity` will be grouped into the **same `Series`**

This is useful when you only know that a set of images belong together, but you don’t have stable DICOM identifiers.

Below we import 3 images with **no `series_instance_uid`**. By setting the same `series_anonymous_identity`, they will be created under **one shared `Series`**.

In [5]:
rows_grouped = [
    ImportRow(
        project_name="anon-series-proj",
        project_external="Y",
        dataset_identifier="",
        patient_identifier="p1",
        study_date=datetime.now().date(),
        storage_backend_key="sb",
        storage_backend_kind="test-kind",
        object_key=f"anon-series-{i}.png",
        series_anonymous_identity=1,  # same => same Series
        modality="ColorFundus",
        modality_tag="anon_series_vendor_tag",
        scan_mode="anon_series_scan",
        manufacturer="m",
        manufacturer_model_name="mm",
        device_description="d",
    )
    for i in range(3)
]

run_group = plan_image_import(session, rows_grouped)
run_group.display_summary()

run_group.apply()
session.commit()

print("Counts after import:")
print_counts(session)

Entity,Update,Create,Total
ImageInstance,,3,3
ImageStorage,,3,3
StorageBackend,,1,1
Scan,,1,1
ModalityTable,,1,1
DeviceModel,,1,1
Project,,1,1
DeviceInstance,,1,1
Patient,,1,1
Study,,1,1


Counts after import:
  Project         : 3
  Patient         : 3
  Study           : 3
  Series          : 6
  ImageInstance   : 8
  ImageStorage    : 8
  ModalityTable   : 3
  Scan            : 3


### Print full change details

If you want a per-entity/per-field view of what will be changed, use `run.full_summary()`.


In [6]:
print(run.full_summary())

Import run f104b86d976c40afaefd476bc7c2811b
Status: done
Change count: 3
1. CREATE Series (SeriesID=5)
  Study : None -> Study (StudyID=2)

2. CREATE ImageInstance (ImageInstanceID=5)
  Series            : None -> Series (SeriesID=5)
  DeviceInstance    : None -> DeviceInstance (DeviceInstanceID=1)
  Scan              : None -> Scan (ScanID=2)
  _Modality         : None -> ModalityTable (ModalityID=2)
  Modality          : None -> Modality.ColorFundus
  Laterality        : None -> Laterality.L
  DatasetIdentifier : None -> 

3. CREATE ImageStorage (ImageStorageID=5)
  StorageBackend : None -> StorageBackend (StorageBackendID=2)
  ImageInstance  : None -> ImageInstance (ImageInstanceID=5)
  ObjectKey      : None -> new_image.png
  Format         : None -> image/png



## Write an import run to JSON

After applying successfully (`run.status == "done"`), you can write the run to JSON.

This JSON is meant for **audit/undo**: it stores entity references by primary key.


In [7]:
out_dir = Path("./_importer_output")
out_dir.mkdir(exist_ok=True)
json_path = out_dir / "import_run.json"

run.write_json(json_path)
print("Wrote:", json_path)

# Inspect the file 
!head -n 20 {json_path}

Wrote: _importer_output/import_run.json
{
  "import_run_id": "f104b86d976c40afaefd476bc7c2811b",
  "status": "done",
  "changes": [
    {
      "type": "create",
      "entity": {
        "type": "entity",
        "entity_model": "Series",
        "primary_key": {
          "SeriesID": 5
        }
      },
      "updates": {
        "Study": {
          "old_value": {
            "type": "value",
            "value": null
          },
          "new_value": {


## Undo an import run (including from a new session)

Undo is often performed later and/or in a different process. The JSON format supports this by resolving entities by primary key.


In [8]:
# Close the current session and open a new one to simulate a separate process.
session.close()
session = SessionLocal()

loaded = ImportRun.read_json(session, json_path)
loaded.display_summary()

print("Counts before undo:")
print_counts(session)   

loaded.undo()
session.commit()

print("Counts after undo:")
print_counts(session)

Entity,Update,Create,Total
Series,,1,1
ImageInstance,,1,1
ImageStorage,,1,1


Counts before undo:
  Project         : 3
  Patient         : 3
  Study           : 3
  Series          : 6
  ImageInstance   : 8
  ImageStorage    : 8
  ModalityTable   : 3
  Scan            : 3
Counts after undo:
  Project         : 3
  Patient         : 3
  Study           : 3
  Series          : 5
  ImageInstance   : 7
  ImageStorage    : 7
  ModalityTable   : 3
  Scan            : 3


## Special use cases

### Add only a Project or a Patient

You don’t need any image fields if you only want to create/patch higher-level entities.

- **Project only**: provide `project_name` (and any project fields)
- **Project + Patient**: add `patient_identifier`


In [9]:
# Project + Patient only
s2 = SessionLocal()

run_pp = plan_image_import(
    s2,
    [
        ImportRow(
            project_name="pp-proj", project_external="Y", patient_identifier="pat-1"
        )
    ],
)
run_pp.display_summary()
run_pp.apply()
s2.commit()

print("Counts:")
print("  Project ", count(s2, Project))
print("  Patient ", count(s2, Patient))
print("  Study   ", count(s2, Study))
print("  Storage ", count(s2, ImageStorage))

Entity,Update,Create,Total
Project,,1,1
Patient,,1,1


Counts:
  Project  4
  Patient  4
  Study    3
  Storage  7


### Update existing entities

You can update an entity by importing a row that resolves to the existing entity (lookup or primary key) and providing fields to change.

Example: import a `Project`, then update its description.


In [10]:
# Fresh DB again for a clean example
SessionLocal2 = create_sqlite_memory_sessionmaker(expire_on_commit=False)

s = SessionLocal2()

run1 = plan_image_import(
    s,
    [
        ImportRow(
            project_name="proj-only",
            project_external="Y",
            project_description="initial",
        )
    ],
)
run1.display_summary()
run1.apply()
s.commit()

run2 = plan_image_import(
    s,
    [ImportRow(project_name="proj-only", project_description="updated")],
)
run2.display_summary()
run2.apply()
s.commit()
print(run2.full_summary())

proj = s.scalar(select(Project))
print("Project:", proj.ProjectID, proj.ProjectName, proj.Description)

Entity,Update,Create,Total
Project,,1,1


Entity,Update,Create,Total
Project,1,,1


Import run b7ccb21e779d4b929cb95c5e41bd3927
Status: done
Change count: 1
1. UPDATE Project (ProjectID=1)
  Description : initial -> updated

Project: 1 proj-only updated


### Update laterality for a few images (identified by `public_id`)

Resolve an existing image by `public_id` (or `sop_instance_uid` / `image_instance_id`) and pass only the fields to change—here, `laterality`.

**Importer pattern** (same as elsewhere in this notebook):

```python
run = plan_image_import(session, [ImportRow(public_id=public_id, laterality="R")])
run.apply()
session.commit()
```

**Rough ORM equivalent** (direct SQLAlchemy; skips import-run bookkeeping):

```python
image = ImageInstance.by_column(session, PublicID=public_id)
image.Laterality = Laterality.R
```

**Why use the importer instead of assigning on the ORM?**

- **Stable API surface**: payload uses **DTO / import field names** (`laterality`, `modality`, `modality_tag`, …), not PascalCase DB columns.
- **One code path**: patches use the **same pipeline** as CSV or `/api/import/image` (validation, mutable vs non-mutable fields, optional parents like Scan / Modality table).
- **Inspect before commit**: `plan_image_import` / `plan_import` support **`display_summary()` / `full_summary()`** before `apply()`.
- **Traceability**: updates are captured as an **`ImportRun`** with field-level old→new, suitable for **logging and auditing**.
- **Undo**: completed runs can be **`undo()`** when paired with stored JSON (see earlier sections).
- **Batching**: update **many rows** and **several fields per row** in one plan (e.g. `laterality` + `modality` + `modality_tag`).
- **Frontend reuse**: the **same import contract** as the HTTP API avoids duplicating patch logic in the viewer.


In [11]:
# 1) Create a few images
rows_seed = [
    ImportRow(
        project_name="lat-proj",
        storage_backend_key="sb-lat",
        object_key=f"lat-{i}.png",
        patient_identifier="pat-lat",
        study_date=datetime(2026, 2, 1).date(),
        modality="ColorFundus",
        laterality="L",
        modality_tag="lat_proj_vendor_tag",
        scan_mode="lat_proj_scan",
    )
    for i in range(3)
]

run_seed = plan_image_import(session, rows_seed, defaults=defaults)
run_seed.display_summary()
run_seed.apply()
session.commit()

print(">>> Before patch")
images = ImageInstance.fetch_all(session)
for im in images:
    print(im.PublicID, im.Laterality)

# 2) Patch laterality by PublicID (no need to restate the whole hierarchy)
rows_patch = [ImportRow(public_id=im.PublicID, laterality="R") for im in images[:5]]

run_patch = plan_image_import(session, rows_patch)
run_patch.display_summary()
print(run_patch.full_summary())

run_patch.apply()
session.commit()

print(">>> After patch")
for im in ImageInstance.fetch_all(session):
    print(im.PublicID, im.Laterality)

Entity,Update,Create,Total
Series,,3,3
ImageInstance,,3,3
ImageStorage,,3,3
StorageBackend,,1,1
Scan,,1,1
ModalityTable,,1,1
Project,,1,1
Patient,,1,1
Study,,1,1


>>> Before patch
nopze7r9bkds Laterality.L
3907jr1kutip Laterality.R
a4ui6tmmzmfm Laterality.L
3a1pcu8r1v6s Laterality.L
x6ogixrq3fwc None
i484mpw2u2jh None
u8blr25pw51v None
1p0ysxq9abti Laterality.L
ub4pvmhexe5m Laterality.L
iu7xr9rzpctl Laterality.L


Entity,Update,Create,Total
ImageInstance,4,,4


Import run ff346ff1a4554240ba3353009d187aa6
Status: pending
Change count: 4
1. UPDATE ImageInstance (ImageInstanceID=1)
  Laterality : Laterality.L -> Laterality.R

2. UPDATE ImageInstance (ImageInstanceID=3)
  Laterality : Laterality.L -> Laterality.R

3. UPDATE ImageInstance (ImageInstanceID=4)
  Laterality : Laterality.L -> Laterality.R

4. UPDATE ImageInstance (ImageInstanceID=6)
  Laterality : None -> Laterality.R

>>> After patch
nopze7r9bkds Laterality.R
3907jr1kutip Laterality.R
a4ui6tmmzmfm Laterality.R
3a1pcu8r1v6s Laterality.R
x6ogixrq3fwc Laterality.R
i484mpw2u2jh None
u8blr25pw51v None
1p0ysxq9abti Laterality.L
ub4pvmhexe5m Laterality.L
iu7xr9rzpctl Laterality.L


### Other behaviors worth knowing (covered by tests)

- **Reject early**: missing required parent chain (e.g. no project identity) fails during planning.
- **Deduplication**: rows with the same lookup key are merged into one create + updates.
- **Idempotency**: re-running the same import should not produce any changes.
- **Primary-key precedence**: if `*_id` is provided, it wins over lookup keys (and allows renaming lookup fields).

See the unit tests under `orm/tests/` for concrete examples.
